In [5]:
import sys
sys.path.append("../../")
from pathlib import Path
from html import escape
import pandas as pd
from IPython.display import HTML, display

from retrieval.product_retriever import search_product
from evaluation.eval import product_load_tests, evaluate_retrieval, context_relevance
from evaluation.classes import ContextRelevanceWrapper
from evaluation.reporting import (
    build_eval_question_report,
    build_retrieval_doc_rows,
    display_eval_question_report,
    export_retrieval_report_to_markdown,
    get_node_text,
)

In [6]:
def retrieving(test, top_k):
    context = search_product(test.question, top_k=top_k)
    nodes = getattr(context, "nodes", context)
    return nodes

In [7]:
async def evaluate_all_retrieval(limit=None, top_k=5, markdown_path=None):
    
    tests = product_load_tests()
    selected_tests = tests[:limit] if limit is not None else tests
    rows = []
    question_reports = []

    for index, test in enumerate(selected_tests, start=1):
        nodes = retrieving(test, top_k)
        
        metrics = evaluate_retrieval(test, nodes, k=top_k)
        
        result = await context_relevance(ContextRelevanceWrapper(question=test.question, contexts=[get_node_text(item) for item in nodes]))
        metrics["context_relevance"] = result
        rows.append(metrics)
        filtered_metrics = {
            key: value
            for key, value in metrics.items()
            if key.startswith("precision") or key.startswith("recall") or key == "mrr" or key.startswith("ndcg") or key == "context_relevance"
        }
        doc_rows = build_retrieval_doc_rows(nodes, test.relevant_docs)
        display_eval_question_report(
            question=test.question,
            relevant_docs=test.relevant_docs,
            metrics=filtered_metrics,
            text_blocks=[],
            doc_rows=doc_rows,
            index=index,
            total=len(selected_tests),
            metric_columns=4,
        )
        question_report = build_eval_question_report(
            question=test.question,
            relevant_docs=test.relevant_docs,
            retrieved_docs=metrics["retrieved_docs"],
            doc_rows=doc_rows,
            extra_fields={"metrics": filtered_metrics},
            index=index,
            total=len(selected_tests),
        )
        question_reports.append(question_report)

    
    report_df = pd.DataFrame(rows)
    metric_columns = [column for column in report_df.columns if column.startswith(("precision", "recall", "ndcg")) or column in {"mrr", "context_relevance"}]
    summary_df = report_df[metric_columns].mean().to_frame(name="mean").T.round(3)

    display(HTML("<h2 style='margin:32px 0 12px;color:#24292f;'>Final metrics</h2>"))
    display(summary_df)

    if markdown_path is not None:
        saved_path = export_retrieval_report_to_markdown(
            report_df,
            question_reports,
            Path(markdown_path),
            notebook_path="notebooks/product/05_product_retrieval_baseline.ipynb",
        )
        display(HTML(f"<p style='margin:12px 0 0;color:#1a7f37;'>Markdown report saved to: <code>{escape(str(saved_path))}</code></p>"))

    return report_df

In [8]:
await evaluate_all_retrieval(markdown_path="../reports/product_retrieving_evaluation_baseline.md", top_k=30, limit=50)

,rank,hit,section_id,title,score,preview
0,1,no,16279046,-,1.0000,Product: 20Dresses Women Blue Trousers Brand: ...
1,2,no,11421530,-,0.8580,Product: Ginni Arora Label Women Red Slim Fit ...
2,3,no,17393956,-,0.8490,Product: Fabindia Women Beige & Red Striped Co...
3,4,no,17378344,-,0.7719,Product: Saffron Threads Women Black Original ...
4,5,no,17625016,-,0.7647,Product: Fabindia Women Off White Cotton Trous...
5,6,no,17104470,-,0.7405,Product: Fabindia Women Blue Striped Relaxed T...
6,7,no,14220222,-,0.6884,Product: DressBerry Women Green Trousers Brand...
7,8,no,18770054,-,0.6864,Product: Trendyol Women Blue & Yellow Ethnic M...
8,9,no,10440556,-,0.6649,Product: DressBerry Women Olive Green Regular ...
9,10,no,15693320,-,0.6387,Product: Fabindia Women Beige Ethnic Trousers ...


,rank,hit,section_id,title,score,preview
0,1,yes,11166548,-,1.0000,Product: Athena Women Burgundy Solid Tailored ...
1,2,no,10758214,-,0.4804,Product: Athena Women Burgundy Solid Pullover ...
2,3,no,12086086,-,0.3919,Product: Athena Women Burgundy Solid Pencil Mi...
3,4,no,11634534,-,0.3760,Product: Athena Women Burgundy Solid Basic Jum...
4,5,no,11634536,-,0.3744,Product: Athena Women Burgundy Solid Basic Jum...
5,6,no,10856152,-,0.3652,Product: SASSAFRAS Women Burgundy Solid Tailor...
6,7,no,11151684,-,0.3112,Product: Athena Women Grey Solid Single Breast...
7,8,no,12402376,-,0.2935,Product: Belle Fille Women Burgundy Solid Asym...
8,9,no,7779375,-,0.2926,Product: Athena Women White Solid Cropped Deni...
9,10,no,13023122,-,0.2605,Product: Athena Women Mustard Yellow Solid Bla...


,rank,hit,section_id,title,score,preview
0,1,yes,18921114,-,1.0000,Product: InWeave Women Red Printed A-Line Skir...
1,2,no,17168254,-,0.9132,Product: InWeave Women Red Printed Pure Cotton...
2,3,no,15322490,-,0.6951,Product: InWeave Women Black & Yellow Lahariya...
3,4,no,17168250,-,0.6593,Product: InWeave Women Blue Printed Pure Cotto...
4,5,no,15542594,-,0.6473,Product: InWeave Women Mustard Yellow & White ...
5,6,no,17168268,-,0.5791,Product: InWeave Women Pink & White Printed Pu...
6,7,no,13497800,-,0.5309,Product: InWeave Women Red Striped A-Line Top ...
7,8,no,16524822,-,0.5103,Product: InWeave Women Striking Pink Solid Asy...
8,9,no,15447990,-,0.4652,Product: InWeave Women Bright Orange Printed T...
9,10,no,15760840,-,0.4117,Product: InWeave Red & Gold-Toned Regular Crop...


,rank,hit,section_id,title,score,preview
0,1,no,10451734,-,1.0000,Product: WEAVERS VILLA Women Mustard Yellow & ...
1,2,no,14402632,-,0.7738,Product: InWeave Mustard Yellow & Blue A-Line ...
2,3,no,19218994,-,0.6543,Product: Mitera Mustard & Red Woven Design Zar...
3,4,no,16719554,-,0.6452,Product: Banarasi Style Mustard Woven Design D...
4,5,no,17663018,-,0.6412,Product: Dupatta Bazaar Mustard Yellow Ethnic ...
5,6,no,1319900,-,0.5850,Product: Desi Weavess Mustard Orange & Red Pri...
6,7,no,11117288,-,0.5546,Product: Mitera Mustard & Pink Silk Blend Wove...
7,8,no,15781530,-,0.4734,Product: max Mustard Viscose Rayon Dupatta Bra...
8,9,no,14088428,-,0.4733,Product: SheWill Mustard Brown & Green Cotton ...
9,10,no,16921550,-,0.4659,Product: Safaa Women Mustard Woven Design Wool...


,rank,hit,section_id,title,score,preview
0,1,no,16124612,-,1.0000,Product: MANGO Black Self Design Regular Top B...
1,2,no,17607938,-,0.8698,Product: MANGO Women Coral Red Solid Double-Br...
2,3,no,15274016,-,0.7830,Product: MANGO Women Black Pullover Brand: MAN...
3,4,no,17808618,-,0.7669,Product: MANGO Off White Satin Finish Extended...
4,5,no,18319598,-,0.7527,Product: MANGO Women Beige Solid Oversize Fit ...
5,6,no,17270200,-,0.6954,Product: MANGO Coral Pink Solid A-line Midi Sk...
6,7,no,14340718,-,0.6601,Product: MANGO Women Peach-Coloured Solid Regu...
7,8,no,16124404,-,0.6342,Product: MANGO Women Cream-Coloured Solid Pull...
8,9,no,15102906,-,0.6028,Product: MANGO Off White Linen Regular Top Bra...
9,10,no,15102850,-,0.5995,Product: MANGO Blue Regular Top Brand: MANGO C...


...

,rank,hit,section_id,title,score,preview
0,1,yes,16835166,-,1.0000,Product: The Roadster Lifestyle Co Women Grey ...
1,2,yes,16835204,-,0.9720,Product: Roadster Women Grey Slim Flared High-...
2,3,yes,16835178,-,0.9379,Product: Roadster Women Grey Slim Flared High-...
3,4,no,14536020,-,0.9170,Product: Roadster Woman Beautiful Grey Solid D...
4,5,yes,13339544,-,0.8984,Product: The Roadster Lifestyle Co Women Grey ...
5,6,no,14954602,-,0.8627,Product: Roadster Women Alluring Grey High-Ris...
6,7,no,14535994,-,0.8579,Product: Roadster Women Grey Solid Denim Truck...
7,8,no,15441284,-,0.7501,Product: The Roadster Lifestyle Co Women Grey ...
8,9,yes,13756948,-,0.7224,Product: Roadster Women Alluring Grey Mid-Rise...
9,10,yes,14954872,-,0.7152,Product: Roadster Women Grey Slouchy Fit Mid-R...


,rank,hit,section_id,title,score,preview
0,1,no,16931960,-,1.0000,Product: all about you Teal Blue Self Design K...
1,2,no,18514120,-,0.9758,Product: Paralians Turquoise Blue Peplum Top B...
2,3,no,14641744,-,0.9617,Product: FabAlley Blue Puff Sleeve Chambray Pe...
3,4,no,11607414,-,0.9172,Product: Bhama Couture Women Blue Printed Pepl...
4,5,no,11607374,-,0.8630,Product: Bhama Couture Women Blue Printed Pepl...
5,6,no,18805426,-,0.7277,Product: BoStreet Blue Checked Peplum Top Bran...
6,7,no,17627582,-,0.6804,Product: kipek Women Deep Navy Blue Solid Top ...
7,8,no,15766290,-,0.6682,Product: Pepe Jeans Women Blue Pure Cotton Tro...
8,9,yes,16504028,-,0.6649,Product: ANVI Be Yourself Women Stunning Blue ...
9,10,no,4374524,-,0.6604,Product: RARE Women Green Self Design Peplum T...


,rank,hit,section_id,title,score,preview
0,1,yes,16533076,-,1.0000,Product: Madame Women Peach-Coloured Self Desi...
1,2,no,18948340,-,0.7722,Product: Madame Peach-Coloured Cuffed Sleeves ...
2,3,no,14556346,-,0.6073,Product: Madame Women White Solid Fuzzy Pullov...
3,4,no,15756110,-,0.5903,Product: Madame Women Pink Pullover Woolen Swe...
4,5,no,16566862,-,0.5359,Product: Madame Women Mauve Lightweight Puffer...
5,6,no,16512732,-,0.5266,Product: Madame Women Beige Self Design Pullov...
6,7,no,15964118,-,0.5018,Product: Madame Women Charcoal Front-Open Swea...
7,8,no,15997770,-,0.4677,Product: Madame Women Yellow & White Checked P...
8,9,no,16512720,-,0.3938,Product: Madame Women Grey Solid Cardigan Swea...
9,10,no,10611424,-,0.3923,Product: Alsace Lorraine Paris Women Peach-Col...


,precision,recall,mrr,ndcg,context_relevance
mean,0.081,0.748,0.57,0.609,0.85


,question,relevant_docs,retrieved_docs,precision,recall,mrr,ndcg,context_relevance
0,Show me Regular trousers within a budget of 2000.,"[17817750, 18770002, 18769968, 18769944, 18769...","[16279046, 11421530, 17393956, 17378344, 17625...",0.033333,0.125,0.050000,0.057588,0.50
1,"Can you find something similar to ""Athena Wome...",[11166548],"[11166548, 10758214, 12086086, 11634534, 11634...",0.033333,1.000,1.000000,1.000000,1.00
2,"Show me products like ""InWeave Women Red Print...",[18921114],"[18921114, 17168254, 15322490, 17168250, 15542...",0.033333,1.000,1.000000,1.000000,1.00
3,I need product with a Woven design pattern in ...,"[12824928, 18262088]","[10451734, 14402632, 19218994, 16719554, 17663...",0.000000,0.000,0.000000,0.000000,1.00
4,I'm looking for products from MANGO for everyd...,"[15315768, 15265896, 15265898, 16892568, 15977...","[16124612, 17607938, 15274016, 17808618, 18319...",0.033333,0.125,0.037037,0.052616,0.50
5,What do you have from Sweet Dreams in Playsuit...,"[11581420, 11581366]","[11581420, 11581448, 11581366, 12882594, 12882...",0.066667,1.000,1.000000,0.919721,1.00
6,I'm looking for Kurta set by Vishudh.,"[13536726, 15997054, 18262138, 13119222, 18263...","[13799826, 18688392, 17656860, 18929144, 17132...",0.266667,1.000,0.250000,0.598079,1.00
7,I'm looking for A-line by Ancestry.,"[15407228, 19152780, 14873702]","[14873856, 15407228, 14873702, 19152780, 19152...",0.100000,1.000,0.500000,0.732829,1.00
8,"Can you find something similar to ""FASHOR Wome...",[19140952],"[19140952, 18964098, 18116634, 19324632, 18964...",0.033333,1.000,1.000000,1.000000,1.00
9,Show me Blue Tailored jacket from Darzi.,[14460680],"[14460680, 14460654, 14460662, 17666226, 17666...",0.033333,1.000,1.000000,1.000000,0.75
